# Feast Operator with RBAC Configuration
## Objective

This demo provides a reference implementation of a runbook on how to enable Role-Based Access Control (RBAC) for Feast using the Feast Operator with the Kubernetes authentication type.

The demo steps include deploying the Feast Operator, creating Feast instances with server components (registry, offline store, online store), and client examples within a Kubernetes environment. The goal is to ensure secure access control for Feast instances deployed by the Feast Operator.
 
Please read these reference documents for understanding the Feast RBAC framework.
- [RBAC Architecture](https://docs.feast.dev/v/master/getting-started/architecture/rbac) 
- [RBAC Permission](https://docs.feast.dev/v/master/getting-started/concepts/permission).
- [RBAC Authorization Manager](https://docs.feast.dev/v/master/getting-started/components/authz_manager)


## Prerequisites
* Kubernetes Cluster
* [kubectl](https://kubernetes.io/docs/tasks/tools/#kubectl) Kubernetes CLI tool.

## Install Prerequisites

The following commands install and configure all the prerequisites on a MacOS environment. You can find the
equivalent instructions on the offical documentation pages:
* Install the `kubectl` cli.
* Install Kubernetes and Container runtime (e.g. [Colima](https://github.com/abiosoft/colima)).
  * Alternatively, authenticate to an existing Kubernetes or OpenShift cluster.
  
```bash
brew install colima kubectl
colima start -r containerd -k -m 3 -d 100 -c 2 --cpu-type max -a x86_64
colima list
```

In [91]:
!kubectl create ns feast
!kubectl config set-context --current --namespace feast


namespace/feast created
Context "kind-kind" modified.


Validate the cluster setup:

In [92]:
!kubectl get ns feast

NAME    STATUS   AGE
feast   Active   12s


## Install the Feast Operator

In [93]:
## Use this install command from a release branch (e.g. 'v0.43-branch')
!kubectl apply -f ../../infra/feast-operator/dist/install.yaml

## OR, for the latest code/builds, use one the following commands from the 'master' branch
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:develop FS_IMG=quay.io/feastdev-ci/feature-server:develop
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:$(git rev-parse HEAD) FS_IMG=quay.io/feastdev-ci/feature-server:$(git rev-parse HEAD)

!kubectl wait --for=condition=available --timeout=5m deployment/feast-operator-controller-manager -n feast-operator-system

namespace/feast-operator-system created
customresourcedefinition.apiextensions.k8s.io/featurestores.feast.dev created
serviceaccount/feast-operator-controller-manager created
role.rbac.authorization.k8s.io/feast-operator-leader-election-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-editor-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-viewer-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-manager-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-auth-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-reader created
rolebinding.rbac.authorization.k8s.io/feast-operator-leader-election-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-manager-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-metrics-auth-rolebinding created
service/feast-operator-controller-manager-metrics-service created
deployment.ap

## Deployment Architecture
The primary objective of this runbook is to guide the deployment of Feast services with RBAC

In this notebook, we will deploy a distributed topology of Feast services, which includes:

* `Registry Server`: Handles metadata storage for feature definitions.
* `Online Store Server`: Uses the `Registry Server` to query metadata and is responsible for low-latency serving of features.
* `Offline Store Server`: Uses the `Registry Server` to query metadata and provides access to batch data for historical feature retrieval.
* `Kubernetes` Authentication types for RBAC Configuration for Feast resources.
* Setting update client RBAC examples to test Feast RBAC based on roles assigned.


## Install the Feast services via FeatureStore CR
Next, we'll use the running Feast Operator to install the feast services with Server components online, offline, registry with kubernetes Authorization set. Apply the included [reference deployment](../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml) to install and configure Feast with kubernetes Authorization .

In [94]:
!cat ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl apply -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml -n feast

apiVersion: feast.dev/v1alpha1
kind: FeatureStore
metadata:
  name: sample-kubernetes-auth
spec:
  feastProject: feast_rbac
  authz:
    kubernetes:
      roles:
        - feast-writer
        - feast-reader
  services:
    offlineStore:
      server: {}
    onlineStore:
      server: {}
    registry:
      local:
        server: {}
    ui: {}
featurestore.feast.dev/sample-kubernetes-auth created


## Validate the running FeatureStore deployment
Validate the deployment status.

In [95]:
!kubectl get all
!kubectl wait --for=condition=available --timeout=8m deployment/feast-sample-kubernetes-auth

NAME                                                READY   STATUS    RESTARTS   AGE
pod/feast-sample-kubernetes-auth-774f6df8df-pck7t   4/4     Running   0          34s

NAME                                            TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)   AGE
service/feast-sample-kubernetes-auth-offline    ClusterIP   10.96.82.193   <none>        80/TCP    34s
service/feast-sample-kubernetes-auth-online     ClusterIP   10.96.21.169   <none>        80/TCP    34s
service/feast-sample-kubernetes-auth-registry   ClusterIP   10.96.115.81   <none>        80/TCP    34s
service/feast-sample-kubernetes-auth-ui         ClusterIP   10.96.57.38    <none>        80/TCP    34s

NAME                                           READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/feast-sample-kubernetes-auth   1/1     1            1           34s

NAME                                                      DESIRED   CURRENT   READY   AGE
replicaset.apps/feast-sample-kubernetes-auth-774f6df8d

Validate that the FeatureStore CR is in a `Ready` state.

In [97]:
!kubectl get feast

NAME                     STATUS   AGE
sample-kubernetes-auth   Ready    80s


## Configure the RBAC Permissions
we have defined permission in `permissions_apply.py`.

In [98]:
#view the permissions  
!cat  permissions_apply.py

from feast.feast_object import ALL_RESOURCE_TYPES
from feast.permissions.action import READ, AuthzedAction, ALL_ACTIONS
from feast.permissions.permission import Permission
from feast.permissions.policy import RoleBasedPolicy

admin_roles = ["feast-writer"]
user_roles = ["feast-reader"]

user_perm = Permission(
    name="feast_user_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=user_roles),
    actions=[AuthzedAction.DESCRIBE] + READ
)

admin_perm = Permission(
    name="feast_admin_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=admin_roles),
    actions=ALL_ACTIONS
)


In [99]:
# Copy the Permissions to the pods under feature_repo directory
!kubectl cp permissions_apply.py $(kubectl get pods -l 'feast.dev/name=sample-kubernetes-auth' -ojsonpath="{.items[*].metadata.name}"):/feast-data/feast_rbac/feature_repo -c online

In [100]:
#view the feature_store.yaml configuration 
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- cat feature_store.yaml

project: feast_rbac
provider: local
offline_store:
    type: dask
online_store:
    path: /feast-data/online_store.db
    type: sqlite
registry:
    path: /feast-data/registry.db
    registry_type: file
auth:
    type: kubernetes
entity_key_serialization_version: 3


## Apply the Permissions and Feast Object to Registry

In [101]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast apply

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/feast-data/feast_rbac/feature_repo/example_repo.py:27: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'driver'.
  driver = Entity(

**List the applied permission details permissions on Feast Resources.**

In [102]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list-roles
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_admin_permission
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_user_permission

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
+--------------+
| ROLE NAME    |
+==============+
| feast-reader |
+--------------+
| feast-writer |
+--------------+
<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>:

## Client Setup & Deploying the Examples

**The Operator creates the client ConfigMap containing the feature_store.yaml. We can retrieve it and store it in the feature_repo.**

In [103]:
!kubectl get configmap feast-sample-kubernetes-auth-client -n feast -o jsonpath='{.data.feature_store\.yaml}' > ./client/feature_repo/feature_store.yaml
!cat client/feature_repo/feature_store.yaml

project: feast_rbac
provider: local
offline_store:
    host: feast-sample-kubernetes-auth-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-sample-kubernetes-auth-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-sample-kubernetes-auth-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: kubernetes
entity_key_serialization_version: 3


In [104]:
#create the client namespace
!kubectl create ns feast-client

namespace/feast-client created


In [105]:
 #ConfigMap for the feature_repo at the client side, if it exists
!kubectl delete configmap client-feature-repo-config --ignore-not-found -n feast-client
!kubectl create configmap client-feature-repo-config --from-file=client/feature_repo -n feast-client

configmap/client-feature-repo-config created


**Deploy the Client Examples**
- As an example, we created 3 different users: 1. [admin_user](client/admin_user_resources.yaml), 2. [readonly_user](client/readonly_user_resources.yaml) and 3. [unauthorized_user](client/unauthorized_user_resources.yaml) .
- Each user is assigned their own service account and roles.

In [106]:
!kubectl apply -f "client/admin_user_resources.yaml"
!kubectl apply -f "client/readonly_user_resources.yaml"
!kubectl apply -f "client/unauthorized_user_resources.yaml"

serviceaccount/feast-admin-sa created
rolebinding.rbac.authorization.k8s.io/feast-admin-rolebinding created
deployment.apps/client-admin-user created
serviceaccount/feast-user-sa created
rolebinding.rbac.authorization.k8s.io/feast-user-rolebinding created
deployment.apps/client-readonly-user created
serviceaccount/feast-unauthorized-user-sa created
deployment.apps/client-unauthorized-user created


## Validate Client

**Run in client-unauthorized-user, unauthorized-user should not able to read or write any resources as no assigned for this user will get the permissions errors as show below.** 

In [107]:
!kubectl exec -n feast-client -it $(kubectl get pods -n feast-client -l app=client-unauthorized-user -o jsonpath="{.items[0].metadata.name}") -- python test.py

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(

--- Historical features for training ---
An error occurred while fetching historical features: Permission error:
Permission feast_admin_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-writer'],Permission feast_user_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-reader']

--- Historical features for batch scoring ---
An error occurred while fetching historical features: Permission error:
Permission feast_admin_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-writ

**Run in client-admin-user. The `admin-user` can perform all actions on all objects.**

In [108]:
!kubectl exec -n feast-client -it $(kubectl get pods -n feast-client -l app=client-admin -o jsonpath="{.items[0].metadata.name}") -- python test.py

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(

--- Historical features for training ---
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.224618
1       1002  ...           20.462153
2       1003  ...           30.089775

[3 rows x 10 columns]

--- Historical features for batch scoring ---
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.786315
1       1002  ...           20.401484
2       1003  ...           30.427706

[3 rows x 10 columns]

--- Load features into online store/materialize_incremental ---
Materializing 2 feature views to 2025-02-20 20:35:39+00:00 into the remote online store.

driver_hourly_stats_fresh from 2025-02

**Run in client-readonly-user,  `readonly-user` can only read or query all objects.**

In [109]:
# Run in client-readonly-user,  `readonly-user` can only read or query all objects.
!kubectl exec -n feast-client -it $(kubectl get pods -n feast-client -l app=client-user -o jsonpath="{.items[0].metadata.name}") -- python test.py

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(

--- Historical features for training ---
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.224618
1       1002  ...           20.462153
2       1003  ...           30.089775

[3 rows x 10 columns]

--- Historical features for batch scoring ---
   driver_id  ... conv_rate_plus_val2
0       1001  ...           10.786315
1       1002  ...           20.401484
2       1003  ...           30.427706

[3 rows x 10 columns]

--- Load features into online store/materialize_incremental ---
Materializing 2 feature views to 2025-02-20 20:37:42+00:00 into the remote online store.

driver_hourly_stats_fresh from 2025-02

## Uninstall the Operator and all related objects

**Uninstall the Operator and all Feast related objects**

In [110]:
!kubectl delete -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl delete -f ../../infra/feast-operator/dist/install.yaml

featurestore.feast.dev "sample-kubernetes-auth" deleted
namespace "feast-operator-system" deleted
customresourcedefinition.apiextensions.k8s.io "featurestores.feast.dev" deleted
serviceaccount "feast-operator-controller-manager" deleted
role.rbac.authorization.k8s.io "feast-operator-leader-election-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-editor-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-viewer-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-manager-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-auth-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-reader" deleted
rolebinding.rbac.authorization.k8s.io "feast-operator-leader-election-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-manager-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-metrics-auth-rolebinding" deleted

**Uninstall Client Related Objects**

In [111]:
!kubectl delete -f client/admin_user_resources.yaml
!kubectl delete -f client/readonly_user_resources.yaml
!kubectl delete -f client/unauthorized_user_resources.yaml
!kubectl delete configmap client-feature-repo-config -n feast-client

serviceaccount "feast-admin-sa" deleted
rolebinding.rbac.authorization.k8s.io "feast-admin-rolebinding" deleted
deployment.apps "client-admin-user" deleted
serviceaccount "feast-user-sa" deleted
rolebinding.rbac.authorization.k8s.io "feast-user-rolebinding" deleted
deployment.apps "client-readonly-user" deleted
serviceaccount "feast-unauthorized-user-sa" deleted
deployment.apps "client-unauthorized-user" deleted
configmap "client-feature-repo-config" deleted


Ensure everything has been removed, or is in the process of being terminated.

In [114]:
!kubectl get all -n feast
!kubectl get all -n feast-client

No resources found in feast namespace.
No resources found in feast-client namespace.
